**Groupe 2**

In [ ]:
# ---------------------------------------------
# 1. Importation des bibliothèques et des données
# ---------------------------------------------
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, precision_recall_curve, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report, auc, roc_curve, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import learning_curve, GridSearchCV
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings

warnings.filterwarnings("ignore")

# Monture de Google Drive
from google.colab import drive
drive.mount('/content/drive')
### BALISE 1
# Chemin du fichier
file_path = '/content/drive/MyDrive/PROJET MACHINE LEARNING/train_2.csv'

df = pd.read_csv(file_path, sep=';')

# ---------------------------------------------
# 2. Définition des fonctions pour le prétraitement
# ---------------------------------------------

# Fonction 1 : Conversion de Vintage en années
def categorize_vintage(df):
    if 'Vintage' not in df.columns:
        return df

    # Remplacer les valeurs nulles ou négatives par une valeur par défaut (par exemple, 0)
    df['Vintage'] = df['Vintage'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)

    # Convertir Vintage en années
    df['Vintage_Years'] = df['Vintage'] / 365

    # Catégorisation de Vintage en intervalles
    max_vintage = df['Vintage_Years'].max()

    # Ensure bins are monotonically increasing
    bins = [0, 0.87,  max(0.87 + 1e-6, max_vintage + 1e-6)]  # Changed this line

    labels = ['< 0.87 years', '> 0.87 years']
    df['Vintage_Cat'] = pd.cut(df['Vintage_Years'], bins=bins, labels=labels, right=True, include_lowest=True)

    return df

def categorize_annual_premium(df):
    max_premium = df['Annual_Premium'].max()
    bins = [0, 10000, 55000, 60000, max_premium]
    # Ensure bins are monotonically increasing
    bins = sorted(list(set(bins)))  # Remove duplicates and sort

    # Adjust labels if necessary
    labels = ['Low', 'Medium', 'High', 'Very High']
    if len(bins) < 5:  # If max_premium is less than 60000
        labels = labels[:len(bins) - 1]

    df['Annual_Premium_Category'] = pd.cut(df['Annual_Premium'], bins=bins, labels=labels, right=False, include_lowest=True, duplicates='drop')
    return df

# Fonction 4 : Transformation logarithmique des primes annuelles
def transform_log_annual_premium(df):
    df['Annual_Premium_Log'] = np.log1p(df['Annual_Premium'])  # Utilisation de log1p pour éviter log(0)
    return df

# Fonction 5 : Catégorisation de l'âge
def categorize_age(df):
    bins = [18, 27, 37, 50, 62, 72, 83, 110]
    labels = ['18-27', '28-37', '38-50', '51-62', '63-72', '73-83', '84-110']
    df['Age_Cat'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True, include_lowest=True)
    return df

# Fonction 6 : Catégorisation de Policy_Sales_Channel
def categorize_int_policy_sales_channel(df, threshold=50):
    channel_counts = df['Policy_Sales_Channel'].value_counts()
    df['Policy_Sales_Channel'] = df['Policy_Sales_Channel'].apply(lambda x: x if channel_counts[x] >= threshold else -1)
    return df

# Fonction 7 : Catégorisation de Region_Code
def categorize_int_region_code(df, threshold=250):
    region_counts = df['Region_Code'].value_counts()
    df['Region_Code'] = df['Region_Code'].apply(lambda x: x if region_counts[x] >= threshold else -1)
    return df

# Fonction 8 : Clustering KMeans (sans Response)
def kmeans_region_policy_premium(df, n_clusters=10):
    columns = ['Region_Code', 'Policy_Sales_Channel', 'Annual_Premium']
    available_columns = [col for col in columns if col in df.columns]
    clustering_data = df[available_columns].fillna(0)  # Gérer les NaN
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    df['Cluster'] = kmeans.fit_predict(clustering_data)
    return df

# Fonction 9 : Encodage des variables catégoriques
def encode_columns(df):
    code_vehicle_damage = {'Yes': 1, 'No': 0}
    code_gender = {'Male': 1, 'Female': 0}
    code_vehicle_age = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}

    df['Vehicle_Damage'] = df['Vehicle_Damage'].map(code_vehicle_damage)
    df['Gender'] = df['Gender'].map(code_gender)
    df['Vehicle_Age'] = df['Vehicle_Age'].map(code_vehicle_age)

    # One-Hot Encoding
    df = pd.get_dummies(df, columns=['Age_Cat', 'Annual_Premium_Category', 'Cluster', 'Vintage_Cat'], drop_first=True)

    return df

# Fonction globale : Prétraitement
def preprocessing(df):
    df = categorize_vintage(df) # met en année et catégorise "> 87" ou "> 87"
    df = categorize_age(df) # categorise l'age
    df = categorize_annual_premium(df) #categorise la prime annuelle
    df = transform_log_annual_premium(df) # transformation logarithmique de la prime
    df = categorize_int_policy_sales_channel(df, threshold=25) # categorise les canaux de distribution
    df = categorize_int_region_code(df, threshold=250) # categorise les regions
    df = kmeans_region_policy_premium(df, n_clusters=10) # on fait un kmeans pour les regions, les canaux, et prime
    df = encode_columns(df) # encodage

    df.drop(['id', 'Age', 'Vintage', 'Annual_Premium', 'Region_Code', 'Policy_Sales_Channel'], axis=1, inplace=True, errors='ignore')
    return df

# ---------------------------------------------
# 3. Préparation des données d'entraînement
# ---------------------------------------------
# Division des données en train/test
train, test = train_test_split(df, test_size=0.2, random_state=42)

X_train = preprocessing(train)
X_test = preprocessing(test)

y_train = X_train['Response']
X_train = X_train.drop('Response', axis=1)

y_test = X_test['Response']
X_test = X_test.drop('Response', axis=1)

# Aligner les colonnes de l'entraînement et du test --> SINON erreur si une modalité apparait dans le testset mais pas dans le trainset
X_train, X_test = X_train.align(X_test, join='left', axis=1)


model = ImbPipeline([
    ('pca', PCA(n_components=0.975)),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('selector', SelectKBest(f_classif, k=15)),
    ('sampler', RandomOverSampler(random_state=42, sampling_strategy=0.9)),  # Étape de sur-échantillonnage
    ('classifier', GradientBoostingClassifier(random_state=42, n_estimators = 50 ))
])


def evaluation_model_without_val(model): # sans validation_score
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  print("F1 score : ", f1_score(y_test, y_pred))
  print('AUC score : ', roc_auc_score(y_test, y_pred))
  print(confusion_matrix(y_test, y_pred))
  print(classification_report(y_test, y_pred))

evaluation_model_without_val(model)

# ---------------------------------------------
# 4. Prediction
# ---------------------------------------------
### BALISE 2
# Charger les données à prédire
pred_file = '/content/drive/MyDrive/PROJET MACHINE LEARNING/test.csv'
X_pred = pd.read_csv(pred_file, sep=',')

# Stocker l'identifiant 'id' avant le prétraitement
id_column = X_pred['id']

# Prétraitement des données à prédire
X_pred = preprocessing(X_pred)
X_pred = X_pred.align(X_train, join='left', axis=1)[0]  # Alignement avec X_train
X_pred.fillna(0, inplace=True)  # Remplir les valeurs manquantes

# Générer les prédictions pour les nouvelles données
y_prob_pred = model.predict_proba(X_pred)[:, 1]  # Probabilités pour la classe positive
best_threshold = 0.75  # Seuil pour la classification
X_pred['Probability'] = y_prob_pred
X_pred['Prediction'] = (y_prob_pred >= best_threshold).astype(int)  # Ajouter les prédictions binaires

# Réintégrer la colonne 'id'
X_pred['id'] = id_column

# Garder uniquement les colonnes id, Probability et Prediction
output_df = X_pred[['id', 'Probability', 'Prediction']]

### BALISE 3
# Sauvegarder les prédictions
output_file = '/content/drive/MyDrive/prediction_2.csv'
output_df.to_csv(output_file, index=False, sep=';')
print(f"Fichier avec prédictions enregistré à : {output_file}")


# ---------------------------------------------
# 5. Fichier CSV
# ---------------------------------------------
### BALISE 3
file_predictions = '/content/drive/MyDrive/prediction_2.csv'

df = pd.read_csv(file_predictions, sep=';')

# On ne garde que les variable id, probability et prediction
df = df[['id', 'Probability', 'Prediction']]

# Combien de prédictions positives et négatives
value_counts = df['Prediction'].value_counts(normalize=True)
print(value_counts)

Mounted at /content/drive
F1 score :  0.6776668114054132
AUC score :  0.8350098372726817
[[5432 2101]
 [ 126 2341]]
              precision    recall  f1-score   support

           0       0.98      0.72      0.83      7533
           1       0.53      0.95      0.68      2467

    accuracy                           0.78     10000
   macro avg       0.75      0.84      0.75     10000
weighted avg       0.87      0.78      0.79     10000

Fichier avec prédictions enregistré à : /content/drive/MyDrive/prediction_2.csv
Prediction
0    0.732167
1    0.267833
Name: proportion, dtype: float64
